In [1]:
import os  # 设置代理环境变量，指向你打通的 1082 端口

os.environ["http_proxy"] = "http://127.0.0.1:1082"
os.environ["https_proxy"] = "http://127.0.0.1:1082"


## RAG中的上下文增强检索
 Retrieval-Augmented Generation（RAG）通过从外部来源检索相关知识来增强AI回复。传统的检索方法返回孤立的文本片段，这可能导致不完整的答案。

为了解决这个问题，我们引入了上下文增强检索，确保检索到的信息包括相邻片段以提高连贯性。

本笔记本中的步骤如下：
- 数据处理：从PDF中提取文本。
- 带有重叠上下文的分块：将文本分割为带有重叠的片段以保留上下文。
- 嵌入创建：将文本片段转换为数值表示。
- 上下文感知检索：检索相关片段及其邻居以提高完整性。
- 回复生成：使用语言模型根据检索到的上下文生成回复。
- 评估：评估模型回复的准确性。

![image.png](./context_enriched_rag.svg)

In [2]:
import fitz  # PyMuPDF，用于处理 PDF 文件
import os  # 操作系统相关功能
import numpy as np  # NumPy，用于数值计算
import json  # JSON 数据处理
from openai import OpenAI  # OpenAI API 客户端

In [3]:
def extract_text_from_pdf(pdf_path):
    """
    从PDF文件中提取文本并打印前`num_chars`个字符。

    参数:
    pdf_path (str): PDF文件的路径。

    返回:
    str: 从PDF中提取的文本。
    """
    # 打开PDF文件
    mypdf = fitz.open(pdf_path)
    all_text = ""  # 初始化一个空字符串用于存储提取的文本

    # 遍历PDF中的每一页
    for page_num in range(mypdf.page_count):
        page = mypdf[page_num]  # 获取页面
        text = page.get_text("text")  # 从页面提取文本
        all_text += text  # 将提取的文本追加到all_text字符串中

    return all_text  # 返回提取的文本


In [4]:
def chunk_text(text, n, overlap):
    """
    将给定的文本分割为长度为 n 的片段，并设置重叠部分。

    参数:
    text (str): 需要分割的文本。
    n (int): 每个片段的字符数。
    overlap (int): 每两个片段之间的重叠字符数。

    返回:
    List[str]: 一个包含文本片段的列表。
    """
    chunks = []  # 初始化一个空列表用于存储片段

    # 使用 (n - overlap) 的步长遍历文本
    for i in range(0, len(text), n - overlap):
        # 将从索引 i 到 i + n 的文本片段追加到 chunks 列表中
        chunks.append(text[i : i + n])

    return chunks  # 返回包含文本片段的列表


In [5]:
from dotenv import load_dotenv
import os

load_dotenv()  # 加载.env文件
api_key = os.getenv("OPENAI_API_KEY")  # 读取密钥

api_key = os.getenv("DASHSCOPE_API_KEY")
client = OpenAI(
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1", api_key=api_key
)


In [6]:
# 定义PDF文件的路径
pdf_path = "data/AI_Information.pdf"

# 从PDF文件中提取文本
extracted_text = extract_text_from_pdf(pdf_path)

# 将提取的文本分割为每段1000个字符且重叠200个字符的片段
text_chunks = chunk_text(extracted_text, 1000, 200)

# 打印生成的文本片段数量
print("Number of text chunks:", len(text_chunks))

# 打印第一个文本片段
print("\nFirst text chunk:")
print(text_chunks[0])


Number of text chunks: 42

First text chunk:
Understanding Artificial Intelligence 
Chapter 1: Introduction to Artificial Intelligence 
Artificial intelligence (AI) refers to the ability of a digital computer or computer-controlled robot 
to perform tasks commonly associated with intelligent beings. The term is frequently applied to 
the project of developing systems endowed with the intellectual processes characteristic of 
humans, such as the ability to reason, discover meaning, generalize, or learn from past 
experience. Over the past few decades, advancements in computing power and data availability 
have significantly accelerated the development and deployment of AI. 
Historical Context 
The idea of artificial intelligence has existed for centuries, often depicted in myths and fiction. 
However, the formal field of AI research began in the mid-20th century. The Dartmouth Workshop 
in 1956 is widely considered the birthplace of AI. Early AI research focused on problem-solving 
and 

In [7]:
def create_embeddings(text_list, model="text-embedding-v4", batch_size=10):
    """
    支持分批处理的嵌入创建函数。
    返回格式兼容后续的 chunk_embedding.embedding 调用。
    """
    # 如果传入的是单个字符串，转为列表统一处理
    if isinstance(text_list, str):
        text_list = [text_list]

    all_data_objects = []

    for i in range(0, len(text_list), batch_size):
        batch = text_list[i : i + batch_size]
        response = client.embeddings.create(model=model, input=batch)

        # 直接将 response.data 中的对象（包含 .embedding 属性）合并
        all_data_objects.extend(response.data)

    # 如果输入是单个字符串，为了匹配原逻辑 create_embeddings(query).data[0]，
    # 我们模拟 OpenAI 的返回结构
    class MockResponse:
        def __init__(self, data):
            self.data = data

    return MockResponse(all_data_objects)


# 调用示例
# 假设 text_chunks 是一个包含 25 个字符串的列表
response = create_embeddings(text_chunks)


## 实现上下文感知语义搜索  
我们对检索进行修改，以包含邻近片段，从而获得更好的上下文信息。

In [8]:
def cosine_similarity(vec1, vec2):
    """
    计算两个向量之间的余弦相似度。

    参数:
    vec1 (np.ndarray): 第一个向量。
    vec2 (np.ndarray): 第二个向量。

    返回:
    float: 两个向量之间的余弦相似度。
    """
    # 计算两个向量的点积并除以其范数的乘积
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))


In [9]:
def context_enriched_search(query, text_chunks,embeddings,k=1,context_size = 1):
    """
    获取最相关的片段及其邻近片段。

    参数:
    query (str): 搜索查询。
    text_chunks (List[str]): 文本片段列表。
    embeddings (List[dict]): 片段嵌入列表。
    k (int): 要检索的相关片段数量。
    context_size (int): 包含的邻近片段数量。

    返回:
    List[str]: 带有上下文信息的相关文本片段。
    """
    # 将查询转换为嵌入向量
    query_embedding = create_embeddings(query).data[0].embedding
    similarity_scores = []

    # 计算查询与每个文本片段嵌入之间的相似度分数
    for i, chunk_embedding in enumerate(embeddings):
        # 计算查询嵌入与当前片段嵌入之间的余弦相似度
        similarity_score = cosine_similarity(
            np.array(query_embedding), np.array(chunk_embedding.embedding)
        )
        # 将索引和相似度分数存储为元组
        similarity_scores.append((i, similarity_score))

    # 按相似度分数降序排序（最高相似度优先）
    similarity_scores.sort(key=lambda x: x[1], reverse=True)

    # 获取最相关片段的索引
    top_index = similarity_scores[0][0]

    # 定义上下文包含的范围
    # 确保不超出文本片段的起始或结束边界
    start = max(0, top_index - context_size)
    end = min(len(text_chunks), top_index + context_size + 1)

    # 返回相关片段及其邻近上下文片段
    return [text_chunks[i] for i in range(start, end)]
    

## 使用上下文检索运行查询  
现在我们测试带有上下文增强的检索。

In [10]:
# 从JSON文件中加载验证数据集
with open("data/val.json") as f:
    data = json.load(f)

# 从数据集中提取第一个问题作为我们的查询
query = data[0]["question"]

# 检索最相关的片段及其上下文片段以提供背景信息
# 参数：
# - query: 我们正在搜索的问题
# - text_chunks: 从PDF中提取的文本片段
# - response.data: 我们的文本片段的嵌入向量
# - k=1: 返回最佳匹配项
# - context_size=1: 在最佳匹配项前后各包含1个片段作为上下文
top_chunks = context_enriched_search(
    query, text_chunks, response.data, k=1, context_size=1
)

# 打印查询以供参考
print("Query:", query)
# 打印每个检索到的片段，并在每个片段前加上标题和分隔符
for i, chunk in enumerate(top_chunks):
    print(f"Context {i + 1}:\n{chunk}\n=====================================")


Query: What is 'Explainable AI' and why is it considered important?
Context 1:
nt aligns with societal values. Education and awareness campaigns inform the public 
about AI, its impacts, and its potential. 
Chapter 19: AI and Ethics 
Principles of Ethical AI 
Ethical AI principles guide the development and deployment of AI systems to ensure they are fair, 
transparent, accountable, and beneficial to society. Key principles include respect for human 
rights, privacy, non-discrimination, and beneficence. 
 
 
Addressing Bias in AI 
AI systems can inherit and amplify biases present in the data they are trained on, leading to unfair 
or discriminatory outcomes. Addressing bias requires careful data collection, algorithm design, 
and ongoing monitoring and evaluation. 
Transparency and Explainability 
Transparency and explainability are essential for building trust in AI systems. Explainable AI (XAI) 
techniques aim to make AI decisions more understandable, enabling users to assess their 
f

## 使用检索到的上下文生成回复  
现在我们使用LLM生成回复。

In [11]:
# 定义AI助手的系统提示
system_prompt = "You are an AI assistant that strictly answers based on the given context. If the answer cannot be derived directly from the provided context, respond with: 'I do not have enough information to answer that.'"


def generate_response(system_prompt, user_message, model="qwen-plus"):
    """
    根据系统提示和用户消息生成AI模型的回复。

    参数:
    system_prompt (str): 用于指导AI行为的系统提示。
    user_message (str): 用户的消息或查询。
    model (str): 用于生成回复的模型。默认为"meta-llama/Llama-2-7B-chat-hf"。

    返回:
    dict: AI模型的回复。
    """
    response = client.chat.completions.create(
        model=model,
        temperature=0,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message},
        ],
    )
    return response


# 根据top片段创建用户提示
user_prompt = "\n".join(
    [
        f"Context {i + 1}:\n{chunk}\n=====================================\n"
        for i, chunk in enumerate(top_chunks)
    ]
)
user_prompt = f"{user_prompt}\nQuestion: {query}"

# 生成AI回复
ai_response = generate_response(system_prompt, user_prompt)
print(ai_response.choices[0].message.content)


Explainable AI (XAI) refers to techniques designed to make AI decisions more understandable to users. It is considered important because it enables users to assess the fairness and accuracy of AI systems, thereby building trust. As highlighted in the contexts, transparency and explainability are essential for understanding AI decision-making processes, evaluating their reliability and fairness, and ensuring accountability—especially in ethically sensitive applications.


## 评估AI回复
我们将AI的回复与预期答案进行比较，并分配一个分数。

In [12]:
# 定义评估系统的系统提示
evaluate_system_prompt = "You are an intelligent evaluation system tasked with assessing the AI assistant's responses. If the AI assistant's response is very close to the true response, assign a score of 1. If the response is incorrect or unsatisfactory in relation to the true response, assign a score of 0. If the response is partially aligned with the true response, assign a score of 0.5."

# 通过组合用户查询、AI回复、真实回复和评估系统提示来创建评估提示
evaluation_prompt = f"User Query: {query}\nAI Response:\n{ai_response.choices[0].message.content}\nTrue Response: {data[0]['ideal_answer']}\n{evaluate_system_prompt}"

# 使用评估系统提示和评估提示生成评估回复
evaluation_response = generate_response(evaluate_system_prompt, evaluation_prompt)

# 打印评估回复
print(evaluation_response.choices[0].message.content)


Score: 1  

Rationale: The AI response is very close to the true response. Both define XAI as aiming for transparency and understandability in AI decision-making, and both emphasize its importance for trust, accountability, and fairness. The AI response adds valuable context—e.g., enabling assessment of fairness/accuracy, supporting reliability evaluation, and highlighting ethical sensitivity—which expands upon but does not contradict the true response. These additions are consistent with standard XAI principles and enhance clarity without introducing inaccuracies. Thus, the core meaning, emphasis, and factual alignment are excellent.
